# KPI 12 — Lead Score vs Order Value

**requirements.md (Lead Quality & Sales Funnel, KPI 12):** "Lead score vs actual order value correlation."

This notebook creates the SQL view `b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value` (idempotent: `CREATE OR REPLACE`). Views are catalog objects, not pipeline datasets — this notebook is **not** attached to `silver_to_gold_etl`; run the cells below once (or after any change to the fact definitions).

**Definitions (state formulas):**

* **Grain:** one row per **company-matched Won lead** (144 rows) — the same attribution unit as KPI 11. Components: 131 valid pairs (score + attributed orders both present — the correlation population), 9 null-score leads with orders, 3 scored leads whose matched company has no orders, 1 both missing. A further 17 Won leads are unmatched (companies not in dim_companies) with no attributable orders — outside the grain, documented here and in the view comment.
* **Order attribution (company bridge):** `fact_leads.company_sk` → `fact_order_items.company_sk`, orders/revenue at **company grain, full order history, no date cut-off** (KPI 11 policy). Each matched company has exactly one Won lead (KPI 11 verification), so no double attribution; SUM over this view equals the KPI 11 attributed totals for net value (9,766,828.34) and net orders (3,147). Gross (10,551,748.55), net (9,766,828.34) and net orders (3,147) all tie to KPI 11 exactly since the 2026-11-20 KPI 11 gross fix.
* **Revenue bases = KPI 1:** gross = all statuses + `order_date_valid`; net = excluding `cancelled` + `order_date_valid`.
* **lead_score_band:** fixed 20-point bands `1-20` … `81-100` (NULL when score is NULL).
* **in_correlation_population:** `lead_score` AND attributed value both present (131 rows); `corr()` ignores NULL rows, so a consumer-side `corr(lead_score, attributed_net_order_value)` over the view reproduces the stored Pearson exactly.
* **Global correlation stats (same value on every row):** `correlation_population_n`, `population_avg_lead_score`, `population_avg_net_order_value`, and `pearson_score_vs_net_value` / `pearson_score_vs_gross_value` / `pearson_score_vs_net_order_count` = ROUND(corr(lead_score, measure), 4) over the 131 valid pairs.
* **attributed_net_value_rank:** ROW_NUMBER by attributed net order value DESC (NULLS LAST), tie-break `lead_id` — the 4 orderless rows rank 141–144.

**KPI 11 basis note (resolved 2026-11-20):** KPI 11's `attributed_gross_revenue` originally summed `line_total` with **no `order_date_valid` filter** — 10,912,463.13, counting the matched companies' 330 invalid-date lines worth 360,714.58 — despite its comment claiming the KPI 1 basis. Fixed later the same day: KPI 11's gross now filters on `order_date_valid` (10,551,748.55), so gross, net and net order count tie exactly between this view and KPI 11 (deep-audited and confirmed).

In [0]:
%sql
-- KPI 12 view: lead score vs actual order value correlation. Idempotent.
CREATE OR REPLACE VIEW b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value
TBLPROPERTIES (
  'comment' = 'KPI 12 - Lead score vs actual order value correlation (requirements.md: Lead Quality & Sales Funnel, KPI 12). Grain: one row per company-matched Won lead (144 rows; KPI 11 attribution unit: fact_leads.company_sk company_name bridge -> fact_order_items.company_sk, orders/revenue at COMPANY grain, full order history, no date cut-off; each matched company has exactly one Won lead so no double attribution). Components: 131 in_correlation_population (score + attributed orders both present), 9 null lead_score with orders, 3 scored with matched company orderless, 1 both missing; 17 further Won leads unmatched (company not in dim_companies) - no attributable orders, excluded. Revenue bases = KPI 1: gross = all statuses + order_date_valid; net = excl cancelled + order_date_valid (all bases tie to KPI 11 exactly since its 2026-11-20 gross fix; KPI 11 gross pre-fix was 10,912,463.13 counting 330 invalid-date lines worth 360,714.58). lead_score_band = fixed 20-point bands 1-20..81-100 (NULL when score NULL). Global stats on every row over the 131 valid pairs: correlation_population_n, population_avg_lead_score, population_avg_net_order_value, pearson_score_vs_net_value/_gross_value/_net_order_count = ROUND(corr(lead_score, measure),4). attributed_net_value_rank = ROW_NUMBER by attributed net order value DESC NULLS LAST (tie-break lead_id). Verified: 144 rows = 144 distinct leads; SUM net 9,766,828.34 / net orders 3,147 = exact KPI 11 attribution match; SUM gross 10,551,748.55 = KPI 1 valid-date basis; 144/144 full match vs independent recompute (correlated-subquery value path + covar_samp/stddev_samp Pearson).'
)
AS
WITH company_orders AS (
  SELECT company_sk,
         SUM(CAST(CASE WHEN order_date_valid THEN line_total ELSE 0 END AS DOUBLE)) AS attributed_gross_order_value,
         SUM(CAST(CASE WHEN order_date_valid AND status <> 'cancelled' THEN line_total ELSE 0 END AS DOUBLE)) AS attributed_net_order_value,
         COUNT(DISTINCT CASE WHEN order_date_valid AND status <> 'cancelled' THEN order_id END) AS attributed_net_order_count
  FROM b2b_ecommerce.gold.fact_order_items
  GROUP BY company_sk
),
matched_won AS (
  SELECT l.lead_id, l.source, l.campaign_name, l.utm_source, l.utm_medium, l.lead_score,
         co.attributed_gross_order_value, co.attributed_net_order_value, co.attributed_net_order_count
  FROM b2b_ecommerce.gold.fact_leads l
  LEFT JOIN company_orders co ON l.company_sk = co.company_sk
  WHERE l.status = 'Won'
    AND l.company_sk IS NOT NULL
),
pop_stats AS (
  SELECT COUNT(*) AS correlation_population_n,
         ROUND(AVG(lead_score), 2) AS population_avg_lead_score,
         ROUND(AVG(attributed_net_order_value), 2) AS population_avg_net_order_value,
         ROUND(corr(lead_score, attributed_net_order_value), 4) AS pearson_score_vs_net_value,
         ROUND(corr(lead_score, attributed_gross_order_value), 4) AS pearson_score_vs_gross_value,
         ROUND(corr(lead_score, attributed_net_order_count), 4) AS pearson_score_vs_net_order_count
  FROM matched_won
  WHERE lead_score IS NOT NULL
    AND attributed_net_order_value IS NOT NULL
)
SELECT m.lead_id,
       m.source,
       m.campaign_name,
       m.utm_source,
       m.utm_medium,
       m.lead_score,
       CASE WHEN m.lead_score IS NULL THEN NULL
            ELSE CONCAT(CAST(FLOOR((m.lead_score - 1) / 20) * 20 + 1 AS STRING), '-',
                        CAST(FLOOR((m.lead_score - 1) / 20) * 20 + 20 AS STRING))
       END AS lead_score_band,
       m.attributed_gross_order_value,
       m.attributed_net_order_value,
       m.attributed_net_order_count,
       CAST(m.lead_score IS NOT NULL AND m.attributed_net_order_value IS NOT NULL AS BOOLEAN) AS in_correlation_population,
       ROW_NUMBER() OVER (ORDER BY m.attributed_net_order_value DESC NULLS LAST, m.lead_id ASC) AS attributed_net_value_rank,
       p.correlation_population_n,
       p.population_avg_lead_score,
       p.population_avg_net_order_value,
       p.pearson_score_vs_net_value,
       p.pearson_score_vs_gross_value,
       p.pearson_score_vs_net_order_count
FROM matched_won m
CROSS JOIN pop_stats p

In [0]:
%sql
-- Verification: shape, totals, integrity. Expected: 144 rows = 144 distinct
-- lead_id (one per company-matched Won lead); components 131 + 9 + 3 + 1
-- (in_correlation_population / null score with orders / scored orderless /
-- both missing); SUM(attributed_net_order_count) = 3,147 / net
-- 9,766,828.34 (exact KPI 11 attribution match); gross 10,551,748.55 =
-- the same KPI 1 valid-date basis KPI 11 uses since its 2026-11-20 fix
-- (pre-fix KPI 11 gross 10,912,463.13 counted 330 invalid-date lines worth
-- 360,714.58);
-- stored Pearson columns = a fresh corr() over the view's population;
-- 0 band / rank / flag / rank-null-pattern mismatches; 0 null keys.
WITH checks AS (
  SELECT v.*,
         CASE WHEN v.lead_score IS NULL THEN NULL
              ELSE CONCAT(CAST(FLOOR((v.lead_score - 1) / 20) * 20 + 1 AS STRING), '-',
                          CAST(FLOOR((v.lead_score - 1) / 20) * 20 + 20 AS STRING))
         END AS band_recompute,
         ROW_NUMBER() OVER (ORDER BY v.attributed_net_order_value DESC NULLS LAST, v.lead_id ASC) AS rank_recompute
  FROM b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value v
)
SELECT
  COUNT(*) AS rows_matched_won,
  COUNT(DISTINCT lead_id) AS distinct_leads,
  SUM(CAST(in_correlation_population AS INT)) AS population_rows,
  SUM(CAST(lead_score IS NULL AND attributed_net_order_value IS NOT NULL AS INT)) AS null_score_with_orders,
  SUM(CAST(lead_score IS NOT NULL AND attributed_net_order_value IS NULL AS INT)) AS scored_orderless,
  SUM(CAST(lead_score IS NULL AND attributed_net_order_value IS NULL AS INT)) AS both_missing,
  SUM(attributed_net_order_count) AS total_net_orders,
  ROUND(SUM(attributed_gross_order_value), 2) AS total_gross_value,
  ROUND(SUM(attributed_net_order_value), 2) AS total_net_value,
  MAX(correlation_population_n) AS population_n,
  MAX(pearson_score_vs_net_value) AS stored_pearson_net,
  MAX(pearson_score_vs_gross_value) AS stored_pearson_gross,
  MAX(pearson_score_vs_net_order_count) AS stored_pearson_orders,
  (SELECT ROUND(corr(lead_score, attributed_net_order_value), 4)
     FROM b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value
    WHERE in_correlation_population) AS fresh_pearson_net,
  (SELECT ROUND(corr(lead_score, attributed_gross_order_value), 4)
     FROM b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value
    WHERE in_correlation_population) AS fresh_pearson_gross,
  (SELECT ROUND(corr(lead_score, attributed_net_order_count), 4)
     FROM b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value
    WHERE in_correlation_population) AS fresh_pearson_orders,
  SUM(CAST(lead_score_band IS DISTINCT FROM band_recompute AS INT)) AS band_mismatches,
  SUM(CAST(attributed_net_value_rank IS DISTINCT FROM rank_recompute AS INT)) AS rank_mismatches,
  SUM(CAST((attributed_net_order_value IS NULL AND attributed_net_value_rank <= 140)
        OR (attributed_net_order_value IS NOT NULL AND attributed_net_value_rank > 140) AS INT)) AS rank_null_pattern_mismatches,
  SUM(CAST(in_correlation_population IS DISTINCT FROM
           CAST(lead_score IS NOT NULL AND attributed_net_order_value IS NOT NULL AS BOOLEAN) AS INT)) AS flag_mismatches,
  SUM(CAST(source IS NULL OR campaign_name IS NULL OR lead_id IS NULL AS INT)) AS null_key_rows
FROM checks

In [0]:
%sql
-- Verification: independently recompute every lead row through a different
-- path - attributed order value via per-lead correlated scalar subqueries on
-- fact_order_items (vs the view's pre-aggregated company CTE join), Pearson
-- via covar_samp / (stddev_samp * stddev_samp) (vs the view's corr()) - then
-- FULL OUTER JOIN against the stored view and compare all per-lead measures.
-- Expected: 144 rows compared, 0 only on either side, 0 mismatch rows;
-- recomputed global Pearson equal to the stored columns (compared at 3 dp to
-- absorb floating-point path differences; both reported at 4 dp).
-- The global_recomputed columns are 1-row constants; MAX() wraps them so
-- the single-row aggregate SELECT parses (otherwise Spark demands GROUP BY).
-- NULL-vs-0 note: COUNT in the correlated-subquery path returns 0 for an
-- orderless company while the view's pre-aggregated CTE join yields NULL
-- (the view's convention: NULL = no attributable orders). The count
-- comparison maps the recomputed 0 back to NULL when the company has no
-- fact rows so the two conventions compare equal.
WITH recomputed AS (
  SELECT l.lead_id, l.source, l.campaign_name, l.utm_source, l.utm_medium, l.lead_score,
         (SELECT SUM(CAST(CASE WHEN oi.order_date_valid THEN oi.line_total ELSE 0 END AS DOUBLE))
            FROM b2b_ecommerce.gold.fact_order_items oi
           WHERE oi.company_sk = l.company_sk) AS gross_recomputed,
         (SELECT SUM(CAST(CASE WHEN oi.order_date_valid AND oi.status <> 'cancelled' THEN oi.line_total ELSE 0 END AS DOUBLE))
            FROM b2b_ecommerce.gold.fact_order_items oi
           WHERE oi.company_sk = l.company_sk) AS net_recomputed,
         (SELECT COUNT(DISTINCT CASE WHEN oi.order_date_valid AND oi.status <> 'cancelled' THEN oi.order_id END)
            FROM b2b_ecommerce.gold.fact_order_items oi
           WHERE oi.company_sk = l.company_sk) AS net_orders_recomputed
  FROM b2b_ecommerce.gold.fact_leads l
  WHERE l.status = 'Won'
    AND l.company_sk IS NOT NULL
),
recomputed_ranked AS (
  SELECT r.*,
         CASE WHEN r.lead_score IS NULL THEN NULL
              ELSE CONCAT(CAST(FLOOR((r.lead_score - 1) / 20) * 20 + 1 AS STRING), '-',
                          CAST(FLOOR((r.lead_score - 1) / 20) * 20 + 20 AS STRING))
         END AS band_recomputed,
         CAST(r.lead_score IS NOT NULL AND r.net_recomputed IS NOT NULL AS BOOLEAN) AS in_population_recomputed,
         ROW_NUMBER() OVER (ORDER BY r.net_recomputed DESC NULLS LAST, r.lead_id ASC) AS rank_recomputed
  FROM recomputed r
),
global_recomputed AS (
  SELECT COUNT(*) AS population_n_recomputed,
         ROUND(covar_samp(lead_score, net_recomputed)
               / (stddev_samp(lead_score) * stddev_samp(net_recomputed)), 4) AS pearson_net_recomputed,
         ROUND(covar_samp(lead_score, gross_recomputed)
               / (stddev_samp(lead_score) * stddev_samp(gross_recomputed)), 4) AS pearson_gross_recomputed,
         ROUND(covar_samp(lead_score, net_orders_recomputed)
               / (stddev_samp(lead_score) * stddev_samp(net_orders_recomputed)), 4) AS pearson_orders_recomputed
  FROM recomputed_ranked
  WHERE lead_score IS NOT NULL
    AND net_recomputed IS NOT NULL
)
SELECT
  SUM(CAST(v.lead_id IS NULL AS INT)) AS only_in_view,
  SUM(CAST(rr.lead_id IS NULL AS INT)) AS only_in_recompute,
  SUM(CAST(v.lead_id IS NOT NULL AND rr.lead_id IS NOT NULL
       AND (v.source IS DISTINCT FROM rr.source
            OR v.campaign_name IS DISTINCT FROM rr.campaign_name
            OR v.utm_source IS DISTINCT FROM rr.utm_source
            OR v.utm_medium IS DISTINCT FROM rr.utm_medium
            OR v.lead_score IS DISTINCT FROM rr.lead_score
            OR v.lead_score_band IS DISTINCT FROM rr.band_recomputed
            OR ROUND(v.attributed_gross_order_value, 2) IS DISTINCT FROM ROUND(rr.gross_recomputed, 2)
            OR ROUND(v.attributed_net_order_value, 2) IS DISTINCT FROM ROUND(rr.net_recomputed, 2)
            OR v.attributed_net_order_count IS DISTINCT FROM
                 CASE WHEN rr.net_recomputed IS NULL THEN NULL ELSE rr.net_orders_recomputed END
            OR v.in_correlation_population IS DISTINCT FROM rr.in_population_recomputed
            OR v.attributed_net_value_rank IS DISTINCT FROM rr.rank_recomputed) AS INT)) AS mismatch_rows,
  MAX(v.correlation_population_n) AS stored_population_n,
  MAX(g.population_n_recomputed) AS population_n_recomputed,
  ROUND(MAX(v.pearson_score_vs_net_value), 4) AS stored_pearson_net,
  MAX(g.pearson_net_recomputed) AS pearson_net_recomputed,
  CAST(ROUND(MAX(v.pearson_score_vs_net_value), 3) IS DISTINCT FROM ROUND(MAX(g.pearson_net_recomputed), 3) AS INT) AS pearson_net_mismatch,
  ROUND(MAX(v.pearson_score_vs_gross_value), 4) AS stored_pearson_gross,
  MAX(g.pearson_gross_recomputed) AS pearson_gross_recomputed,
  CAST(ROUND(MAX(v.pearson_score_vs_gross_value), 3) IS DISTINCT FROM ROUND(MAX(g.pearson_gross_recomputed), 3) AS INT) AS pearson_gross_mismatch,
  ROUND(MAX(v.pearson_score_vs_net_order_count), 4) AS stored_pearson_orders,
  MAX(g.pearson_orders_recomputed) AS pearson_orders_recomputed,
  CAST(ROUND(MAX(v.pearson_score_vs_net_order_count), 3) IS DISTINCT FROM ROUND(MAX(g.pearson_orders_recomputed), 3) AS INT) AS pearson_orders_mismatch
FROM b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value v
FULL OUTER JOIN recomputed_ranked rr ON v.lead_id = rr.lead_id
CROSS JOIN global_recomputed g

In [0]:
%sql
-- Headline: score-band gradient over the scored leads, with the global
-- correlation stats (constant on every row, surfaced via MAX). The 10
-- null-score leads (NULL band) sit in the view but are excluded here.
SELECT lead_score_band,
       COUNT(*) AS leads,
       ROUND(AVG(lead_score), 1) AS avg_lead_score,
       ROUND(AVG(attributed_net_order_value), 2) AS avg_net_order_value,
       ROUND(SUM(attributed_net_order_value), 2) AS total_net_order_value,
       ROUND(AVG(attributed_net_order_count), 1) AS avg_net_orders,
       MAX(correlation_population_n) AS correlation_population_n,
       MAX(population_avg_lead_score) AS population_avg_lead_score,
       MAX(population_avg_net_order_value) AS population_avg_net_order_value,
       MAX(pearson_score_vs_net_value) AS pearson_score_vs_net_value,
       MAX(pearson_score_vs_gross_value) AS pearson_score_vs_gross_value,
       MAX(pearson_score_vs_net_order_count) AS pearson_score_vs_net_order_count
FROM b2b_ecommerce.gold.kpi_12_lead_score_vs_order_value
WHERE lead_score_band IS NOT NULL
GROUP BY lead_score_band
ORDER BY lead_score_band